In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

# pandas and numpy are used for data handling and numeric operations
import pandas as pd
import numpy as np

# joblib is used to save the trained model to disk
import joblib

# scikit-learn utilities for splitting data and building preprocessing pipelines
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Regression algorithms to compare
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Evaluation metrics to compare model performance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# 2. LOAD DATASET
# ============================================================

# Load the salary prediction dataset from a CSV file
# Update the path below if the file is stored in a different location
df = pd.read_csv(
    r"C:\Users\choud\Downloads\job_salary_prediction_dataset.csv"
)

# Show the first few rows to verify the data loaded correctly
print(df.head())


# ============================================================
# 3. UNDERSTAND THE DATASET
# ============================================================

# Inspect dataset shape (rows, columns)
print("Dataset Shape:")
print(df.shape)

# Display data types and non-null counts
print("\nDataset Information:")
df.info()

# Check for missing values in each column
print("\nMissing Values:")
print(df.isnull().sum())

# Check for duplicate rows before training
print("\nDuplicate Rows:")
print(df.duplicated().sum())

# Display basic statistical summary for numeric columns
print("\nStatistical Summary:")
print(df.describe())


# ============================================================
# 4. DATA CLEANING
# ============================================================

# Remove duplicate rows to avoid biased training
# (Duplicate rows do not provide additional information)
df = df.drop_duplicates()

# Missing values are handled later inside the pipeline using SimpleImputer.
# This preserves the original dataset structure and prevents data leakage.


# ============================================================
# 5. FEATURE SELECTION
# ============================================================

# Define the target column for prediction
# The model will learn to predict the salary value
target = "salary"

# Separate independent variables from the target column
X = df.drop(columns=[target])
y = df[target]


# ============================================================
# 6. IDENTIFY CATEGORICAL AND NUMERICAL FEATURES
# ============================================================

# Detect categorical columns by data type
categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# Detect numeric columns by data type
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("\nCategorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)


# ============================================================
# 7. TRAIN/TEST SPLIT
# ============================================================

# Split the dataset into training and testing sets
# 20% of data is reserved for evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTraining Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)


# ============================================================
# 8. CREATE PREPROCESSING PIPELINES
# ============================================================

# Pipeline for numeric features: impute missing values then scale
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

# Pipeline for categorical features: impute missing values then one-hot encode
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine the numeric and categorical pipelines
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])


# ============================================================
# 9. DEFINE THREE REGRESSION MODELS
# ============================================================

# Set up three different regression models for comparison
models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=15,
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=30,
        max_depth=15,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    )
}

# ============================================================
# 10. TRAIN AND EVALUATE ALL MODELS
# ============================================================

results = []
trained_models = {}

for model_name, model in models.items():
    print(f"\nTraining {model_name}...")

    # Build a full pipeline that first preprocesses features then fits the model
    model_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", model)
    ])

    # Train the pipeline on the training data
    model_pipeline.fit(X_train, y_train)

    # Predict salaries on the test set
    y_pred = model_pipeline.predict(X_test)

    # Compute evaluation metrics for this model
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    })

    trained_models[model_name] = model_pipeline

    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R2 Score:", r2)


# ============================================================
# 11. COMPARE MODEL PERFORMANCE
# ============================================================

# Create a DataFrame to compare the metric scores for each model
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="R2 Score",
    ascending=False
)

print("\nModel Comparison:")
print(results_df)


# ============================================================
# 12. SELECT THE BEST MODEL
# ============================================================

# Pick the model with the highest R2 score as the best performer
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print("\nBest Model:", best_model_name)


# ============================================================
# 13. SAVE THE BEST MODEL
# ============================================================

# Persist the best pipeline to disk for later use
joblib.dump(
    best_model,
    "best_salary_prediction_model.pkl"
)

print("\nBest model saved successfully!")

            job_title  experience_years education_level  skills_count  \
0         AI Engineer                10        Bachelor             2   
1        Data Analyst                 5        Bachelor            17   
2  Frontend Developer                18             PhD             4   
3    Business Analyst                19             PhD            13   
4     Product Manager                15        Bachelor             7   

        industry company_size   location remote_work  certifications  salary  
0     Healthcare       Medium      India      Hybrid               2  109413  
1        Telecom        Small  Australia          No               0   93764  
2          Media       Medium  Singapore          No               1  148123  
3         Retail       Medium     Canada         Yes               0  189123  
4  Manufacturing        Large     Sweden         Yes               0  165069  
Dataset Shape:
(250000, 10)

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex

C:\Users\choud\AppData\Local\Temp\ipykernel_16132\2161857669.py:86: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(



Categorical Features:
['job_title', 'education_level', 'industry', 'company_size', 'location', 'remote_work']

Numerical Features:
['experience_years', 'skills_count', 'certifications']

Training Data Shape: (200000, 9)
Testing Data Shape: (50000, 9)

Training Linear Regression...
MAE: 5436.0969586494775
RMSE: 7125.522920575376
R2 Score: 0.9634690226760201

Training Decision Tree...
MAE: 8689.49892652206
RMSE: 10982.677456375672
R2 Score: 0.913215080117932

Training Random Forest...
MAE: 7710.9437418135885
RMSE: 9812.881035780312
R2 Score: 0.9307179266533128

Model Comparison:
               Model          MAE          RMSE  R2 Score
0  Linear Regression  5436.096959   7125.522921  0.963469
2      Random Forest  7710.943742   9812.881036  0.930718
1      Decision Tree  8689.498927  10982.677456  0.913215

Best Model: Linear Regression

Best model saved successfully!
